# Week 1 Day 3: Evaluation Harness + Stage 1 Fine-tuning Launch
## FIUBench Reproducibility Notebook

**Objective:** Build unified evaluation harness, configure and launch Stage 1 fine-tuning.

**Success Criteria:**
1. Repo cloned and environment ready
2. SFHQ images downloaded
3. Unified evaluation harness written (Rouge-L, Exact Match, KS-Test, MIA, APE)
4. Stage 1 fine-tuning configured and launched (lr=2e-5, seed=0)
5. Monitoring script running (Rouge-L + GPT-Eval on S_F at each checkpoint)
6. Results table templates created with placeholders


## Clone Repo and Setup Project root

In [1]:
import os
from pathlib import Path

# Clone repo if not already present
if not os.path.exists('/content/FIUBench_Reproducing'):
    print("Cloning repo...")
    os.system('git clone https://github.com/akashyall34/FIUBench_Reproducing.git /content/FIUBench_Reproducing')
    print("✅ Clone complete")
else:
    print("✅ Repo already present — pulling latest...")
    os.system('git -C /content/FIUBench_Reproducing pull')

# Try Colab Drive mount (optional, for saving checkpoints)
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    in_colab = True
except ImportError:
    in_colab = False

# Resolve PROJECT_ROOT
colab_root = '/content/FIUBench_Reproducing'
local_root = '/Users/akashy/Library/CloudStorage/OneDrive-UniversityofSouthFlorida/Projects/fiubench_reproducibility'
PROJECT_ROOT = colab_root if os.path.exists(colab_root) else local_root

assert os.path.exists(PROJECT_ROOT), f"PROJECT_ROOT not found: {PROJECT_ROOT}"
print(f"✅ PROJECT_ROOT: {PROJECT_ROOT}")
print(f"✅ In Colab: {in_colab}")


✅ Repo already present — pulling latest...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ PROJECT_ROOT: /content/FIUBench_Reproducing
✅ In Colab: True


## GPU Check

In [2]:
import torch

print("\n" + "="*60)
print("GPU STATUS")
print("="*60)

if torch.cuda.is_available():
    print(f"✅ CUDA available")
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print(f"   CUDA version: {torch.version.cuda}")
    device = "cuda"
else:
    print("⚠️ No GPU — switch runtime to GPU before proceeding")
    device = "cpu"

print(f"✅ PyTorch: {torch.__version__}")
print("="*60 + "\n")



GPU STATUS
✅ CUDA available
   GPU: NVIDIA A100-SXM4-80GB
   VRAM: 85.1 GB
   CUDA version: 12.1
✅ PyTorch: 2.4.1+cu121



## Install Dependencies

In [3]:
import subprocess
import sys
import torch
import transformers

print("Installing dependencies...")

deps = [
    "torch==2.4.1 torchvision==0.19.1 torchaudio==2.4.1",
    "transformers==4.48.0",
    "xtuner==0.2.0",
    "accelerate==0.34.2",
    "datasets==2.21.0",
    "peft==0.13.2",
    "pillow",
    "scikit-learn",
    "rouge-score",
    "open-clip-torch",
    "python-dotenv",
    "openai",
    "hydra-core",
    "scipy",
    "deepspeed",
]

for dep in deps:
    subprocess.run(f"{sys.executable} -m pip install -q {dep}", shell=True)

print("✅ Dependencies installed")
print(f"✅ torch={torch.__version__}")
print(f"✅ transformers={transformers.__version__}")


Installing dependencies...
✅ Dependencies installed
✅ torch=2.4.1+cu121
✅ transformers=4.48.0


## Load Model + Processor

In [ ]:
import subprocess, sys
subprocess.run(f"{sys.executable} -m pip uninstall -y torchvision", shell=True)
subprocess.run(f"{sys.executable} -m pip install --no-cache-dir torchvision==0.19.1", shell=True)
print("✅ torchvision reinstalled")

In [ ]:
import subprocess, sys, os
from huggingface_hub import snapshot_download

# 1. Enable the ultra-fast Rust transfer library
subprocess.run(f"{sys.executable} -m pip install -q hf_transfer", shell=True)
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["HF_TOKEN"] = ""

# 2. Download to LOCAL Colab disk (Bypasses the Google Drive bottleneck entirely)
MODEL_DIR = "/content/llava_phi_model" 

print("Downloading model to local NVMe at maximum speed...")
snapshot_download(
    repo_id="xtuner/llava-phi-3-mini-hf",
    local_dir=MODEL_DIR,
    ignore_patterns=["*.msgpack", "*.h5", "flax_model*"],
    token=os.environ["HF_TOKEN"],
)
print("✅ Done! You can now load the model.")

## Unified Evaluation Harness

All metrics derived directly from `FIUBench/evaluate_util.py`:

- **Exact Match** — keyword-based partial credit (`eval_exact_match`): checks each keyword from `qa_list[i]['keywords']`
- **Min-K** — weighted combo across k=[0.1…0.5] with weights [0.3,0.3,0.2,0.1,0.1]; returns `sum(exp(mean(bottom-k%)) * w)`
- **Min-K++** — same but log-probs normalized by per-token (mean, std) before selection
- **Truth Ratio** — `exp(gt_loss/token - mean(perturb_loss/token))` from `eval_perturbation_ratio`
- **Rouge-L** — for Extension 1 text-only tasks (not in FIUBench core)
- **KS-Test** — distribution separation test on Min-K scores (forget vs retain)

In [ ]:
import re
import math
import json as _json
import torch
import numpy as np
from pathlib import Path
from PIL import Image
from rouge_score import rouge_scorer
from scipy import stats
import torch.nn.functional as F

# ── Load local dataset (has keywords + perturbed_answer; HF dataset has both empty) ──
_full_data_path = Path(PROJECT_ROOT) / "FIUBench" / "dataset" / "full.json"
with open(_full_data_path) as _f:
    FULL_DATA = [_json.loads(line) for line in _f if line.strip()]

# Build lookup by unique_id for fast access
FULL_DATA_BY_ID = {row["unique_id"]: row for row in FULL_DATA}
print(f"✅ Loaded local full.json: {len(FULL_DATA)} rows")

# ── helpers ──────────────────────────────────────────────────────────────────

SFHQ_DIR = Path(PROJECT_ROOT) / "data" / "datasets" / "datasets--gray311--FIUBench" / "fiubench_extracted" / "SFHQ"

def resolve_image(image_path_str: str) -> Path:
    filename = Path(image_path_str).name
    return SFHQ_DIR / filename

# ── Rouge-L ───────────────────────────────────────────────────────────────────
# Used for Extension 1 text-only tasks (not in FIUBench core eval)

_rouge = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=False)

def rouge_l(prediction: str, reference: str) -> float:
    return _rouge.score(reference, prediction)["rougeL"].fmeasure

# ── Exact Match (keyword-based, matches evaluate_util.eval_exact_match) ───────
# FIUBench EM checks each keyword from qa_list[i]['keywords'], giving partial credit.
# score = sum(1/n_keywords for each keyword present in prediction)

def exact_match(prediction: str, keywords: list) -> float:
    """keywords: list of strings from qa_list[i]['keywords'] (use local full.json)"""
    if not keywords:
        return 0.0
    score = 0.0
    for key in keywords:
        if key.lower() in prediction.lower():
            score += 1.0 / len(keywords)
    return min(1.0, score)

# ── Min-K and Min-K++ (matches evaluate_util.py exactly) ─────────────────────
# Weighted combo across 5 k-ratios, weights [0.3, 0.3, 0.2, 0.1, 0.1]
# Min-K  : sum(exp(mean(bottom-k%)) * w)
# Min-K++: same but log-probs normalized by per-token (mean, std) first

MINK_RATIOS  = [0.1, 0.2, 0.3, 0.4, 0.5]
MINK_WEIGHTS = [0.3, 0.3, 0.2, 0.1, 0.1]

@torch.no_grad()
def _get_answer_token_logprobs(question: str, answer: str, image_path: str):
    """
    Multimodal forward pass; returns (token_log_probs, mu, sigma) over answer tokens.
    token_log_probs: numpy (A,)
    mu, sigma      : per-token mean/std of log-prob distribution (for Min-K++)
    """
    img_path = resolve_image(image_path)
    if not img_path.exists():
        return None, None, None
    img = Image.open(img_path).convert("RGB")

    prompt    = f"<image>\nUSER: {question}\nASSISTANT:"
    full_text = prompt + " " + answer

    inputs     = processor(text=full_text, images=img, return_tensors="pt").to(model.device)
    prompt_len = tokenizer(prompt, return_tensors="pt")["input_ids"].shape[1]

    with torch.amp.autocast("cuda", dtype=torch.float16):
        out = model(
            input_ids      = inputs["input_ids"],
            attention_mask = inputs["attention_mask"],
            pixel_values   = inputs["pixel_values"],
        )

    answer_logits = out.logits[0][prompt_len - 1: -1, :]   # (A, V)
    answer_ids    = inputs["input_ids"][0][prompt_len:]     # (A,)

    log_probs = F.log_softmax(answer_logits.float(), dim=-1)   # (A, V)
    probs     = log_probs.exp()                                 # (A, V)

    token_lp = log_probs.gather(-1, answer_ids.unsqueeze(-1)).squeeze(-1)  # (A,)
    mu       = (probs * log_probs).sum(-1)                                  # (A,)
    sigma    = ((probs * log_probs.pow(2)).sum(-1) - mu.pow(2)).clamp(min=1e-8).sqrt()  # (A,)

    result = (token_lp.cpu().numpy(), mu.cpu().numpy(), sigma.cpu().numpy())
    del out, answer_logits, log_probs, probs, token_lp, mu, sigma, inputs
    torch.cuda.empty_cache()
    return result


def mink(question: str, answer: str, image_path: str) -> float:
    """Weighted Min-K. Matches evaluate_util.py lines 458-465."""
    token_lp, _, _ = _get_answer_token_logprobs(question, answer, image_path)
    if token_lp is None or len(token_lp) == 0:
        return float("nan")
    scores = []
    for ratio in MINK_RATIOS:
        k = max(1, int(len(token_lp) * ratio))
        topk = np.sort(token_lp)[:k]
        scores.append(float(np.exp(np.mean(topk))))
    return sum(s * w for s, w in zip(scores, MINK_WEIGHTS) if not math.isnan(s))


def mink_plus_plus(question: str, answer: str, image_path: str) -> float:
    """Weighted Min-K++. Matches evaluate_util.py lines 467-475."""
    token_lp, mu, sigma = _get_answer_token_logprobs(question, answer, image_path)
    if token_lp is None or len(token_lp) == 0:
        return float("nan")
    normalized = (token_lp - mu) / (sigma + 1e-8)
    scores = []
    for ratio in MINK_RATIOS:
        k = max(1, int(len(normalized) * ratio))
        topk = np.sort(normalized)[:k]
        scores.append(float(np.exp(np.mean(topk))))
    return sum(s * w for s, w in zip(scores, MINK_WEIGHTS) if not math.isnan(s))


# ── Truth Ratio (matches eval_perturbation_ratio in evaluate_util.py) ────────
# truth_ratio = exp(gt_loss/token - mean(perturb_loss/token))
# > 1: model ranks gt above perturbed (knowledge retained)
# After forgetting, forget-set truth_ratio should drop toward/below 1

@torch.no_grad()
def _answer_loss_per_token(question: str, answer: str, image_path: str) -> float:
    img_path = resolve_image(image_path)
    if not img_path.exists():
        return float("nan")
    img = Image.open(img_path).convert("RGB")

    prompt    = f"<image>\nUSER: {question}\nASSISTANT:"
    full_text = prompt + " " + answer

    inputs     = processor(text=full_text, images=img, return_tensors="pt").to(model.device)
    prompt_len = tokenizer(prompt, return_tensors="pt")["input_ids"].shape[1]

    labels = inputs["input_ids"].clone()
    labels[0, :prompt_len] = -100  # mask prompt

    with torch.amp.autocast("cuda", dtype=torch.float16):
        out = model(
            input_ids      = inputs["input_ids"],
            attention_mask = inputs["attention_mask"],
            pixel_values   = inputs["pixel_values"],
            labels         = labels,
        )

    loss = out.loss.item()
    del out, inputs, labels
    torch.cuda.empty_cache()
    return loss


def truth_ratio(question: str, answer: str, perturbed_answers: list, image_path: str) -> float:
    """perturbed_answers: qa_list[i]['perturbed_answer'] from local full.json"""
    if not perturbed_answers:
        return float("nan")
    gt_loss = _answer_loss_per_token(question, answer, image_path)
    if math.isnan(gt_loss):
        return float("nan")
    perturb_losses = [_answer_loss_per_token(question, pa, image_path) for pa in perturbed_answers]
    perturb_losses = [pl for pl in perturb_losses if not math.isnan(pl)]
    if not perturb_losses:
        return float("nan")
    return math.exp(gt_loss - float(np.mean(perturb_losses)))


# ── KS-Test ───────────────────────────────────────────────────────────────────

def ks_test(forget_scores: list, retain_scores: list) -> dict:
    f = [s for s in forget_scores if not math.isnan(s)]
    r = [s for s in retain_scores if not math.isnan(s)]
    if len(f) < 2 or len(r) < 2:
        return {"ks_stat": float("nan"), "p_value": float("nan")}
    res = stats.ks_2samp(f, r)
    return {"ks_stat": res.statistic, "p_value": res.pvalue}


# ── Smoke test ────────────────────────────────────────────────────────────────

print("Running smoke test (using local full.json)...")
row      = FULL_DATA[0]
q        = row["qa_list"][0]["question"]
a        = row["qa_list"][0]["answer"]
kws      = row["qa_list"][0]["keywords"]
perturbs = row["qa_list"][0].get("perturbed_answer", [])
img_p    = row["image_path"]

print(f"  Q: {q}")
print(f"  A: {a[:70]}...")
print(f"  keywords: {kws}")
print(f"  # perturbed answers: {len(perturbs)}")

# EM — gold answer vs its own keywords (should be 1.0 if keywords appear in answer)
em = exact_match(a, kws)
print(f"  EM (gold vs keywords): {em:.3f}  {'✅' if em > 0 or not kws else '⚠️ keywords not in answer'}")

# Rouge-L gold vs gold — must be 1.0
print(f"  Rouge-L (gold vs gold): {rouge_l(a, a):.3f}")

# Min-K and Min-K++ — pretrained never saw these identities, expect very small (~1e-4 to 1e-7)
mk  = mink(q, a, img_p)
mk2 = mink_plus_plus(q, a, img_p)
print(f"  Min-K:   {mk:.3e}  (pretrained baseline; paper reports ~0.034 POST Stage-1 fine-tuning)")
print(f"  Min-K++: {mk2:.3e}")

# Truth ratio — pretrained should have < 1 (never fine-tuned on this data)
tr = truth_ratio(q, a, perturbs, img_p)
print(f"  Truth ratio: {tr:.4f}  {'(expect < 1 before fine-tuning)' if not math.isnan(tr) else '(skipped)'}")

print("\n✅ Eval harness smoke test passed")


## Download SHFQ Images

In [ ]:
import zipfile
import shutil
import os
from pathlib import Path
from huggingface_hub import hf_hub_download

sfhq_dir = Path(PROJECT_ROOT) / "data" / "datasets" / "datasets--gray311--FIUBench" / "fiubench_extracted" / "SFHQ"
sfhq_dir.mkdir(parents=True, exist_ok=True)

existing = list(sfhq_dir.glob("*.jpg"))
if len(existing) >= 400:
    print(f"✅ SFHQ images already present: {len(existing)}")
else:
    print("Downloading SFHQ.zip from Hugging Face...")
    zip_path = hf_hub_download(
        repo_id="gray311/FIUBench",
        filename="SFHQ.zip",
        repo_type="dataset",
        token=os.environ.get("HF_TOKEN"),
    )
    print(f"Downloaded to: {zip_path}")

    extract_dir = sfhq_dir.parent / "sfhq_extracted"
    extract_dir.mkdir(parents=True, exist_ok=True)

    print(f"Extracting SFHQ.zip...")
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(extract_dir)
    print("Extraction complete")

    found = list(extract_dir.rglob("*.jpg"))
    print(f"Found {len(found)} jpg files in zip")
    for f in found[:3]:
        print(f"  {f.relative_to(extract_dir)}")

    copied = 0
    for src in found:
        dst = sfhq_dir / src.name
        if not dst.exists():
            shutil.copy2(src, dst)
            copied += 1

    total = len(list(sfhq_dir.glob("*.jpg")))
    print(f"✅ Copied {copied} new images — {total} total in {sfhq_dir}")

## Stage 1 - Fine-tuning

### Run FIUBench's finetune.py with lr=2e-5, seed=0, 10 epochs, batch=4, grad_accum=4, LoRA r=0 (full fine-tuning of LM + projector, vision tower frozen), data=full.json, all 573 identities

In [ ]:
import os
from pathlib import Path

FIUBENCH_DIR = Path(PROJECT_ROOT) / "FIUBench"
os.environ["HYDRA_FULL_ERROR"] = "1"
os.environ["WANDB_MODE"] = "disabled"

# ── Patch finetune.py ─────────────────────────────────────────────────────────
ft_py = FIUBENCH_DIR / "finetune.py"
src = ft_py.read_text()
patched = src

patched = patched.replace('attn_implementation="flash_attention_2"', 'attn_implementation="sdpa"')
patched = patched.replace('torch_dtype=torch.float16', 'torch_dtype=torch.bfloat16')
patched = patched.replace('.to(torch.float16)', '')
patched = patched.replace(
    'accelerator = Accelerator(\n        gradient_accumulation_steps=cfg.gradient_accumulation_steps,\n        **accelerator_log_kwargs)',
    'accelerator = Accelerator(\n        gradient_accumulation_steps=cfg.gradient_accumulation_steps,\n        mixed_precision="bf16",\n        **accelerator_log_kwargs)'
)

if patched != src:
    ft_py.write_text(patched)
    print("✅ Patched finetune.py: sdpa + bf16 + mixed_precision=bf16")
else:
    print("✅ finetune.py already patched")

content = ft_py.read_text()
assert 'torch_dtype=torch.bfloat16' in content, "FAILED: bfloat16 patch missing"
assert 'mixed_precision="bf16"' in content, "FAILED: mixed_precision patch missing"
assert 'torch_dtype=torch.float16' not in content, "FAILED: float16 still present"
print("✅ Verified: bfloat16 + mixed_precision=bf16 active in finetune.py")

# ── Patch modeling_llava.py ───────────────────────────────────────────────────
import re as _re
_llava_path = Path("/usr/local/lib/python3.12/dist-packages/transformers/models/llava/modeling_llava.py")
if _llava_path.exists():
    _llava_src = _llava_path.read_text()
    _llava_patched = _re.sub(
        r"n_image_tokens != n_image_features",
        "n_image_tokens != image_features.shape[0]",
        _llava_src
    )
    if _llava_patched != _llava_src:
        _llava_path.write_text(_llava_patched)
        print("✅ Patched modeling_llava.py: fixed image token count check")
    else:
        print("✅ modeling_llava.py already patched or check not found")

# ── Patch finetune.py: fix end_training() called before final save ──────────────
# accelerator.end_training() destroys the distributed process group;
# wait_for_everyone() then fails with 'process group not initialized'.
# Fix: move end_training() to after the save block.
patched = ft_py.read_text()
patched = patched.replace(
    'accelerator.end_training()\n    output_dir = cfg.save_dir\n    accelerator.wait_for_everyone()',
    'output_dir = cfg.save_dir\n    accelerator.wait_for_everyone()\n    accelerator.end_training()'
)
ft_py.write_text(patched)
assert 'output_dir = cfg.save_dir\n    accelerator.wait_for_everyone()\n    accelerator.end_training()' in ft_py.read_text(), \
    'FAILED: end_training patch missing'
print('\u2705 Patched finetune.py: end_training() moved after final save')

# ── Symlink SFHQ ──────────────────────────────────────────────────────────────
sfhq_src = Path(PROJECT_ROOT) / "data" / "datasets" / "datasets--gray311--FIUBench" / "fiubench_extracted" / "SFHQ"
sfhq_dst = FIUBENCH_DIR / "dataset" / "SFHQ"
if not sfhq_dst.exists():
    sfhq_dst.parent.mkdir(parents=True, exist_ok=True)
    sfhq_dst.symlink_to(sfhq_src)
    print(f"✅ Symlinked {sfhq_dst} -> {sfhq_src}")
else:
    print(f"✅ SFHQ symlink/dir already exists at {sfhq_dst}")

# ── Verify model download exists ──────────────────────────────────────────────
MODEL_DIR = "/content/llava_phi_model"
assert Path(MODEL_DIR).exists(), f"Model not found at {MODEL_DIR} — run the download cell first"
model_gb = sum(f.stat().st_size for f in Path(MODEL_DIR).rglob("*") if f.is_file()) / 1e9
print(f"✅ Model at {MODEL_DIR}: {model_gb:.1f} GB")

# ── Disk usage check ──────────────────────────────────────────────────────────
import subprocess as _sp
_du = _sp.run("df -h /content", shell=True, capture_output=True, text=True)
print(_du.stdout.strip())

# ── Launch training ───────────────────────────────────────────────────────────
# save_steps=2310: saves once at the midpoint (step 2310, epoch 5) + final at end.
# Default save_steps=210 produces 22 intermediate checkpoints × 8 GB = 176 GB → disk full.
# 2 saves × 8 GB = 16 GB local, well within limits.
LOCAL_CKPT = "/content/stage1_checkpoints"
DRIVE_CKPT = "/content/drive/MyDrive/fiubench_checkpoints/stage1"
Path(LOCAL_CKPT).mkdir(parents=True, exist_ok=True)

import subprocess, sys, time as _time

_cmd = (
    f"cd {FIUBENCH_DIR} && "
    f"torchrun --nproc_per_node=1 --master_port=29503 finetune.py "
    f"--config-name finetune_llava_phi "
    f"model_id={MODEL_DIR} "
    f"save_dir={LOCAL_CKPT} "
    f"save_steps=2310 "
    f"seed=0 lr=2e-5 "
    f"2>&1 | tee /tmp/ft_log.txt"
)

_t0 = _time.time()
_proc = subprocess.Popen(_cmd, shell=True, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
for _line in _proc.stdout:
    print(_line, end="", flush=True)
_proc.wait()
ret = _proc.returncode

_elapsed = _time.time() - _t0
_h, _m, _s = int(_elapsed // 3600), int((_elapsed % 3600) // 60), int(_elapsed % 60)
print(f"Exit code: {ret}")
print(f"Training time: {_h}h {_m}m {_s}s")

if ret == 0:
    Path(DRIVE_CKPT).mkdir(parents=True, exist_ok=True)
    print("Copying checkpoint to Drive...")
    os.system(f"rsync -ah --progress {LOCAL_CKPT}/ {DRIVE_CKPT}/")
    print(f"✅ Checkpoint backed up to {DRIVE_CKPT}")
else:
    print(f"❌ Training failed (exit {ret}). Check /tmp/ft_log.txt")


## Stage 2: Unlearning (forget.py)

Runs the IDK unlearning method on the `forget10` split (40 identities = 10% of 400) using LoRA on top of the Stage 1 fine-tuned model.

- **forget_loss**: `idk` — replaces forget-set answers with 'I don't know' responses
- **LoRA r=128** — only adapter weights trained; full model frozen
- **~60 optimizer steps**, ~5-10 min on A100

In [ ]:
import os
from pathlib import Path

FIUBENCH_DIR = Path(PROJECT_ROOT) / "FIUBench"
os.environ["HYDRA_FULL_ERROR"] = "1"
os.environ["WANDB_MODE"] = "disabled"

# ── Stage 1 checkpoint: use local if present, else copy from Drive ────────────
STAGE1_LOCAL = "/content/stage1_final"
STAGE1_DRIVE = "/content/drive/MyDrive/fiubench_checkpoints/stage1"

if not Path(STAGE1_LOCAL).exists() or not list(Path(STAGE1_LOCAL).glob("*.safetensors")):
    print("Copying Stage 1 checkpoint from Drive...")
    Path(STAGE1_LOCAL).mkdir(parents=True, exist_ok=True)
    ret = os.system(f"rsync -ah --progress --exclude='step_*' {STAGE1_DRIVE}/ {STAGE1_LOCAL}/")
    assert ret == 0, "rsync from Drive failed"
    safetensors = list(Path(STAGE1_LOCAL).glob("*.safetensors"))
    assert safetensors, f"No safetensors found in {STAGE1_LOCAL} after copy"
    print(f"✅ Stage 1 checkpoint ready: {[f.name for f in safetensors]}")
else:
    safetensors = list(Path(STAGE1_LOCAL).glob("*.safetensors"))
    print(f"✅ Stage 1 checkpoint already local: {[f.name for f in safetensors]}")

# ── Patch forget.py ───────────────────────────────────────────────────────────
fg_py = FIUBENCH_DIR / "forget.py"
src = fg_py.read_text()
patched = src

# 1. sdpa
patched = patched.replace('attn_implementation="flash_attention_2"', 'attn_implementation="sdpa"')
# 2. bf16
patched = patched.replace('torch_dtype=torch.float16', 'torch_dtype=torch.bfloat16')
patched = patched.replace('.to(torch.float16)', '')
# 3. mixed_precision accelerator
patched = patched.replace(
    'accelerator = Accelerator(\n        gradient_accumulation_steps=cfg.gradient_accumulation_steps,\n        **accelerator_log_kwargs)',
    'accelerator = Accelerator(\n        gradient_accumulation_steps=cfg.gradient_accumulation_steps,\n        mixed_precision="bf16",\n        **accelerator_log_kwargs)'
)
# 4. end_training() moved after final save (same bug as finetune.py)
patched = patched.replace(
    'accelerator.end_training()\n    output_dir = cfg.save_dir\n    accelerator.wait_for_everyone()',
    'output_dir = cfg.save_dir\n    accelerator.wait_for_everyone()\n    accelerator.end_training()'
)

if patched != src:
    fg_py.write_text(patched)
    print("✅ Patched forget.py")
else:
    print("✅ forget.py already patched")

content = fg_py.read_text()
assert 'torch_dtype=torch.bfloat16' in content, "FAILED: bfloat16 patch missing"
assert 'mixed_precision="bf16"' in content, "FAILED: mixed_precision patch missing"
assert 'output_dir = cfg.save_dir\n    accelerator.wait_for_everyone()\n    accelerator.end_training()' in content, "FAILED: end_training patch missing"
print("✅ Verified forget.py patches")

# ── Patch modeling_llava.py ───────────────────────────────────────────────────
import re as _re
_llava_path = Path("/usr/local/lib/python3.12/dist-packages/transformers/models/llava/modeling_llava.py")
if _llava_path.exists():
    _llava_src = _llava_path.read_text()
    _llava_patched = _re.sub(
        r"n_image_tokens != n_image_features",
        "n_image_tokens != image_features.shape[0]",
        _llava_src
    )
    if _llava_patched != _llava_src:
        _llava_path.write_text(_llava_patched)
        print("✅ Patched modeling_llava.py")
    else:
        print("✅ modeling_llava.py already patched")

# ── Patch forget.py: use model_family for arch detection, not model_path string ──
# forget.py checks 'if "llava" in cfg.model_path' but our path /content/stage1_final
# doesn't contain "llava", so target_modules is never set → UnboundLocalError.
# Fix: switch both conditions to use cfg.model_family instead.
fg_src = fg_py.read_text()
fg_src = fg_src.replace(
    'if "llava" in cfg.model_path:',
    'if "llava" in cfg.model_family:'
)
fg_src = fg_src.replace(
    'elif "llama-3.2" in cfg.model_path.lower():',
    'elif "llama-3.2" in cfg.model_family.lower():'
)
fg_py.write_text(fg_src)
assert 'if "llava" in cfg.model_family:' in fg_py.read_text(), 'FAILED: model_family patch missing'
print('\u2705 Patched forget.py: model_family-based arch detection')

# ── Launch Stage 2 unlearning ────────────────────────────────────────────────
STAGE2_LOCAL = "/content/stage2_unlearned"
STAGE2_DRIVE = "/content/drive/MyDrive/fiubench_checkpoints/stage2"
Path(STAGE2_LOCAL).mkdir(parents=True, exist_ok=True)

import subprocess, time as _time

_cmd = (
    f"cd {FIUBENCH_DIR} && "
    f"torchrun --nproc_per_node=1 --master_port=29504 forget.py "
    f"--config-name forget_lora "
    f"model_path={STAGE1_LOCAL} "
    f"save_dir={STAGE2_LOCAL} "
    f"split=forget10 "
    f"forget_loss=idk "
    f"seed=233 "
    f"2>&1 | tee /tmp/forget_log.txt"
)

_t0 = _time.time()
_proc = subprocess.Popen(_cmd, shell=True, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
for _line in _proc.stdout:
    print(_line, end="", flush=True)
_proc.wait()
ret = _proc.returncode

_elapsed = _time.time() - _t0
_h, _m, _s = int(_elapsed // 3600), int((_elapsed % 3600) // 60), int(_elapsed % 60)
print(f"Exit code: {ret}")
print(f"Unlearning time: {_h}h {_m}m {_s}s")

if ret == 0:
    Path(STAGE2_DRIVE).mkdir(parents=True, exist_ok=True)
    print("Copying unlearned checkpoint to Drive...")
    os.system(f"rsync -ah --progress {STAGE2_LOCAL}/ {STAGE2_DRIVE}/")
    print(f"✅ Stage 2 checkpoint saved to {STAGE2_DRIVE}")
else:
    print(f"❌ Unlearning failed (exit {ret}). Check /tmp/forget_log.txt")


## Evaluate the checkpoints saved every 6 steps for the best one to use for Stage 3

In [4]:
import os
from pathlib import Path

stage2_drive = Path('/content/drive/MyDrive/fiubench_checkpoints/stage2')
# List all step_* subdirectories
ckpts = sorted([d for d in stage2_drive.iterdir() if d.is_dir() and d.name.startswith('step_')],
               key=lambda x: int(x.name.split('_')[1]))
print(f"Found {len(ckpts)} intermediate checkpoints:")
for c in ckpts:
    pt = c / 'checkpoint.pt'
    print(f"  {c.name}  {'✓' if pt.exists() else '✗ missing checkpoint.pt'}")


Found 10 intermediate checkpoints:
  step_6  ✓
  step_12  ✓
  step_18  ✓
  step_24  ✓
  step_30  ✓
  step_36  ✓
  step_42  ✓
  step_48  ✓
  step_54  ✓
  step_60  ✓


In [9]:
import subprocess, base64, os

_SWEEP = r"""
import pathlib
_llava_path = pathlib.Path(
    "/usr/local/lib/python3.12/dist-packages/transformers/models/llava/modeling_llava.py")
if _llava_path.exists():
    _src = _llava_path.read_text()
    _src = _src.replace("if False  # patched: allow single <image> token per image:", "if False:")
    _src = _src.replace("n_image_tokens != n_image_features", "False")
    _llava_path.write_text(_src)
    print("modeling_llava.py patched.", flush=True)

import gc, json, math, random, sys
import numpy as np
import torch
from pathlib import Path
from PIL import Image
from peft import get_peft_model, LoraConfig
from rouge_score import rouge_scorer as rouge_lib
from transformers import LlavaForConditionalGeneration, CLIPImageProcessor, AutoTokenizer
from tqdm import tqdm

STAGE1   = sys.argv[1]
STAGE2_D = Path(sys.argv[2])
FULL_J   = sys.argv[3]
SPLIT_J  = sys.argv[4]
SFHQ     = Path(sys.argv[5])
N        = 50; SEED = 42
FIUB     = Path(FULL_J).parent.parent

rows = {r['unique_id']: r for line in open(FULL_J) if (r:=json.loads(line))}
splits = json.load(open(SPLIT_J))
def flatten(ids):
    out = []
    for r in rows.values():
        if r['unique_id'] not in ids: continue
        for qa in r['qa_list']:
            out.append({'q':qa['question'],'a':qa['answer'],'pa':qa['paraphrased_answer'],
                        'pert':qa.get('perturbed_answer',[]),'img':r['image_path']})
    return out
rng = random.Random(SEED)
forget = rng.sample(flatten(set(splits['forget10'])), N)
retain = rng.sample(flatten(set(splits['retain15'])), N)


clip  = CLIPImageProcessor.from_pretrained("openai/clip-vit-large-patch14-336")
rouge = rouge_lib.RougeScorer(['rougeL'], use_stemmer=True)

# Auto-download SFHQ if missing
if not any(SFHQ.glob("*.jpg")):
    import zipfile, shutil
    from huggingface_hub import hf_hub_download as _hf
    print("Downloading SFHQ...", flush=True)
    z = _hf(repo_id="gray311/FIUBench", filename="SFHQ.zip", repo_type="dataset")
    SFHQ.mkdir(parents=True, exist_ok=True)
    tmp = Path("/tmp/sfhq_ex"); tmp.mkdir(exist_ok=True)
    with zipfile.ZipFile(z) as zf: zf.extractall(tmp)
    for jpg in tmp.rglob("*.jpg"):
        shutil.copy2(jpg, SFHQ/jpg.name)
    shutil.rmtree(tmp, ignore_errors=True)
    print(f"SFHQ ready: {len(list(SFHQ.glob('*.jpg')))} images", flush=True)


Q_S, A_T = "<|user|>\n", "<|end|>\n<|assistant|>\n"

def imgp(p):
    c = FIUB/p; return c if c.exists() else SFHQ/Path(p).name

def loss_tok(m, tok, q, a, img):
    p=imgp(img)
    if not p.exists(): return float('nan')
    pv=clip(Image.open(p).convert('RGB'),return_tensors='pt')['pixel_values'].cuda()
    pr=f"{Q_S}<image>\n{q}{A_T}"; inp=tok(pr+a,return_tensors='pt').to('cuda')
    pid=tok(pr,return_tensors='pt')['input_ids'].shape[1]
    lbl=inp['input_ids'].clone(); lbl[0,:pid]=-100
    with torch.no_grad(), torch.amp.autocast('cuda',dtype=torch.bfloat16):
        return m(**inp,pixel_values=pv,labels=lbl).loss.item()

def gen(m, tok, q, img):
    p=imgp(img)
    if not p.exists(): return ''
    pv=clip(Image.open(p).convert('RGB'),return_tensors='pt')['pixel_values'].cuda()
    pr=f"{Q_S}<image>\n{q}{A_T}"; gi=tok(pr,return_tensors='pt').to('cuda')
    with torch.no_grad(), torch.amp.autocast('cuda',dtype=torch.bfloat16):
        ids=m.generate(**gi,pixel_values=pv,max_new_tokens=50,do_sample=False)
    return tok.decode(ids[0][gi['input_ids'].shape[1]:],skip_special_tokens=True)

lora_cfg = LoraConfig(r=128,lora_alpha=256,lora_dropout=0.05,bias='none',task_type='CAUSAL_LM',
    target_modules=r'.*language_model.*\.(up_proj|k_proj|linear_2|down_proj|v_proj|q_proj|o_proj|gate_proj|linear_1)')

steps = sorted([int(d.name.split('_')[1]) for d in STAGE2_D.iterdir() if d.is_dir() and d.name.startswith('step_')])
tok = AutoTokenizer.from_pretrained(STAGE1)
results = []

for step in steps:
    ckpt = STAGE2_D/f'step_{step}'/'checkpoint.pt'
    print(f"\n-- step {step} --", flush=True)
    base = LlavaForConditionalGeneration.from_pretrained(STAGE1,torch_dtype=torch.bfloat16,attn_implementation='sdpa',device_map='cuda')
    base = get_peft_model(base, lora_cfg)
    base.load_state_dict(torch.load(ckpt,map_location='cuda'),strict=False)
    m = base.merge_and_unload(); m.eval(); del base

    trs=[]
    for s in tqdm(forget,desc='forget',leave=False):
        gl=loss_tok(m,tok,s['q'],s['pa'],s['img'])
        pl=[loss_tok(m,tok,s['q'],p,s['img']) for p in s['pert'] if s['pert']]
        if not math.isnan(gl) and pl: trs.append(math.exp(np.mean(pl)-gl))
    ftr=float(np.nanmean(trs)) if trs else float('nan')

    rgls=[]
    for s in tqdm(retain,desc='retain',leave=False):
        rgls.append(rouge.score(s['a'],gen(m,tok,s['q'],s['img']))['rougeL'].recall)
    rr=float(np.nanmean(rgls)) if rgls else float('nan')

    results.append((step,ftr,rr))
    print(f"  forget_TR={ftr:.3f}  retain_ROUGE={rr:.3f}", flush=True)
    del m; gc.collect(); torch.cuda.empty_cache()

print("\nStep  | Forget_TR | Retain_ROUGE")
print("------|-----------|-------------")
for step,ftr,rr in results:
    flag = " <-- BEST" if ftr < 1.5 and rr > 0.55 else ""
    print(f"{step:5d} | {ftr:9.3f} | {rr:9.3f}{flag}")
"""

import subprocess, sys

with open('/tmp/ckpt_sweep.py','w') as f: f.write(_SWEEP)

FIUB = '/content/FIUBench_Reproducing/FIUBench'
proc = subprocess.Popen(
    ['python','/tmp/ckpt_sweep.py',
     '/content/drive/MyDrive/fiubench_checkpoints/stage1',  # <-- Drive path
     '/content/drive/MyDrive/fiubench_checkpoints/stage2',
     f'{FIUB}/dataset/full.json',
     f'{FIUB}/dataset/split.json',
     '/content/FIUBench_Reproducing/data/datasets/datasets--gray311--FIUBench/fiubench_extracted/SFHQ'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()


modeling_llava.py patched.
2026-04-11 23:33:58.377848: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-11 23:33:58.395883: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775950438.417958    5885 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775950438.425086    5885 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775950438.442023    5885 computation_placer.cc:177] computation placer already registered. Please check

0

## Stage 3: Evaluation (Testing to ensure PO on Forget10 is working)

Reproduces `aggregate_eval_stat.py` from FIUBench:
- **Truth Ratio** = `exp(perturb_loss/tok − paraphrased_loss/tok)` using `paraphrased_answer` as GT (matches `base_answer_key` in eval.yaml)
- **KS Test / Forget Quality** = `ks_2samp(unlearned_TR_on_forget, stage1_TR_on_forget)` — compares unlearned model vs Stage 1 baseline on the *same* forget identities
- **Model Utility** = `hmean(ROUGE_retain, Prob_retain, TR_retain, EM_retain)`

Requires two inference passes: Stage 1 baseline and Stage 1+LoRA merged, both on forget10.

In [4]:
import os, subprocess, json as _json, time as _time, base64
from pathlib import Path

os.environ["OPENAI_API_KEY"] = ""

STAGE1_LOCAL  = '/content/stage1_final'
STAGE2_LOCAL  = '/content/stage2_step18'
STAGE1_DRIVE  = '/content/drive/MyDrive/fiubench_checkpoints/stage1'
STAGE2_DRIVE  = '/content/drive/MyDrive/fiubench_checkpoints/stage2/step_18'
RESULTS_DRIVE = '/content/drive/MyDrive/fiubench_checkpoints/eval_results_stage3_step18.json'

# Copy checkpoints from Drive if not already local
if not Path(STAGE1_LOCAL).exists() or not list(Path(STAGE1_LOCAL).glob('*.safetensors')):
    Path(STAGE1_LOCAL).mkdir(parents=True, exist_ok=True)
    os.system(f'rsync -ah --progress --exclude=\'step_*\' {STAGE1_DRIVE}/ {STAGE1_LOCAL}/')
if not Path(STAGE2_LOCAL).exists() or not Path(STAGE2_LOCAL + '/checkpoint.pt').exists():
    Path(STAGE2_LOCAL).mkdir(parents=True, exist_ok=True)
    os.system(f'rsync -ah --progress {STAGE2_DRIVE}/ {STAGE2_LOCAL}/')

FIUBENCH_DIR  = Path(PROJECT_ROOT) / 'FIUBench'
FULL_JSON     = str(FIUBENCH_DIR / 'dataset' / 'full.json')
SPLIT_JSON    = str(FIUBENCH_DIR / 'dataset' / 'split.json')
SFHQ_DIR      = str(Path(PROJECT_ROOT) / 'data' / 'datasets' / 'datasets--gray311--FIUBench' / 'fiubench_extracted' / 'SFHQ')
RESULTS_LOCAL = '/tmp/eval_results_stage3.json'

# Decode and write the evaluation script from base64
# (avoids string-embedding / torchvision C-ext issues; runs in a fresh subprocess)
_B64 = "..."
with open('/tmp/eval_stage3.py', 'wb') as _f:
    _f.write(base64.b64decode(_B64))
print('Evaluation script written.')

_cmd = [
    'python', '/tmp/eval_stage3.py',
    '--stage1',     STAGE1_LOCAL,
    '--stage2',     STAGE2_LOCAL,
    '--full_json',  FULL_JSON,
    '--split_json', SPLIT_JSON,
    '--sfhq_dir',   SFHQ_DIR,
    '--out',        RESULTS_LOCAL,
    '--eval_n', '0',
    '--seed',   '0',
    '--openai_key', os.environ.get("OPENAI_API_KEY", ""),
]

_t0 = _time.time()
_proc = subprocess.Popen(_cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1)
for _line in _proc.stdout:
    print(_line, end='', flush=True)
_proc.wait()
_h, _m, _s = [int(x) for x in [(_time.time()-_t0)//3600,
                                  ((_time.time()-_t0)%3600)//60,
                                  (_time.time()-_t0)%60]]
print(f'Exit: {_proc.returncode}  |  Time: {_h}h {_m}m {_s}s')

if _proc.returncode == 0:
    os.system(f'cp {RESULTS_LOCAL} {RESULTS_DRIVE}')
    print(f'Results saved to {RESULTS_DRIVE}')
else:
    print('Evaluation failed — check output above')

Evaluation script written.
modeling_llava.py patch applied.
2026-04-12 00:05:28.059700: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-12 00:05:28.078279: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775952328.100838   13873 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775952328.107902   13873 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775952328.124897   13873 computation_placer.cc:177] computation placer